In [ ]:
# Project 2: Grazioso Salvare dashboard
# Joshua Mertz

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash import ctx
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "cs340project"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will return a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Img(
        src='data:image/png;base64,{}'.format(encoded_image.decode()),
        style={'width': '200px', 'height': 'auto'}
    ),
    html.P("Contact: joshua.mertz@snhu.edu"),  # unique identifier
    html.Hr(),
    html.Div([
        dcc.Dropdown(['---', 'Water', 'Mountain; Wilderness', 'Disaster; Individual Tracking'], '---', id='dropdown'),
    ]),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        editable=False,
        row_selectable="single",  # selectable rows
        filter_action="native",
        sort_action="native",  # sortable
        selected_rows=[0],  # first row selected by default
        page_action="native",  # basic pagination
        page_current=0,
        page_size=10  # set to 5 in the screenshot for legibility
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(
        className='row',
        style={'display' : 'flex'},
        children=[
            html.Div(id='map-id', className='col s12 m6'),
            html.Div(id='graph-id', className='col s12 m6')
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################
# dropdown menu
@app.callback(
    Output('datatable-id', 'data'),
    Input('dropdown', 'value')
)
def update_dashboard(value):
    df = pd.DataFrame.from_records(db.read({}))
    if (value == "Water"):
        df = pd.DataFrame.from_records(db.read({
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": { "$gt": 26, "$lt": 156 }
        }))
    elif (value == "Mountain; Wilderness"):
        df = pd.DataFrame.from_records(db.read({
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": { "$gt": 26, "$lt": 156 }
        }))
    elif (value == "Disaster; Individual Tracking"):
        df = pd.DataFrame.from_records(db.read({
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": { "$gt": 20, "$lt": 300 }
        }))
    # start of function already reads in all records
    # therefore no special operation needed for btnSubmitReset    
    df.drop(columns=["_id"], inplace=True)
    return df.to_dict("records")

    #        columns=[{"na2me": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
    #        data=df.to_dict('records')
    #        return (data,columns)

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_graphs(viewData, index):
    dff = pd.DataFrame.from_dict(viewData)
    # Because only single row selection allowed, the list can be converted to a row index
    if not index:
        row = 0
    else: 
        row = index[0]

    return [
       dcc.Graph(
           style={'width': '550px', 'height': '300px'},
           figure = px.pie(dff, names='outcome_type', title='Outcome Types')
       )
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:  # guard against None selected
        return []
    return [{'if': {'column_id': i}, 'background_color': '#D2F3FF'} for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):    
    dff = pd.DataFrame.from_dict(viewData)
    # Because only single row selection allowed, the list can be converted to a row index
    if not index:
        row = 0
    else: 
        row = index[0]
        
    coordinate_pos = [dff.iloc[row,13], dff.iloc[row,14]]

    return [dl.Map(
        style={'width': '550px', 'height': '300px'},
        center=coordinate_pos,  # Austin TX is at [30.75,-97.48]
        zoom=10,
        children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(
                position=coordinate_pos,
                children=[dl.Tooltip(dff.iloc[row,4]), 
                          dl.Popup([html.H1("Animal Name"), html.P(dff.iloc[row,9])])]
            )
        ]
    )]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode="external", port=8054)